# Community Notes 陣営反応分析 — v2 (control 拡張版)

上から順にセルを実行してください。
「★ 設定」セルのパラメータを変えることで実行時間やトピックを調整できます。

**v2 で増えたもの**
- `ratings_count` (評価総数, 人気度の control)
- `bridging_score` (評価者 polarity 多様性, X 公開 MF noteScore の自前近似)
  - 各ノートの評価者 polarity の `Var(x) + Var(y)` で計算 (data 不要・既存 polarity_df を再利用)
- M0 (旧式) / M1 (+評価数) / M2 (+評価数+bridging) の 3 モデルを比較

**結果の読み方**
- M2 で β_TypeA が **残れば** 「polarity 多様性を揃えてもなお陣営攻撃の効果あり (アルゴリズムの穴)」
- M2 で β_TypeA が **消えれば** 「polarity 多様性不足で説明できる (CN アルゴリズム仕様の帰結)」

> **品質モデルについて**: `models/quality_model.joblib` が repo にあれば自動使用 (LLM ラベル学習済み)。

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ チャンク実行設定 (全データを少量ずつ分けて処理)
# ====================================================
FILES_PER_CHUNK  = 2
ROWS_PER_CHUNK   = 500_000
FILE_CHUNK_INDEX = 0        # ← 0, 1, 2, 3
ROW_CHUNK_INDEX  = 0        # ← 0から順に増やす

FILE_OFFSET   = FILE_CHUNK_INDEX * FILES_PER_CHUNK
SKIP_ROWS     = ROW_CHUNK_INDEX  * ROWS_PER_CHUNK
CHUNK_SUFFIX  = f'_f{FILE_CHUNK_INDEX}_r{ROW_CHUNK_INDEX}'
RATINGS_FILES = FILES_PER_CHUNK
NROWS         = ROWS_PER_CHUNK

DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'
SAVE_DIR   = '/content/drive/MyDrive/toriumi_x3_results_v2'

TOPICS = {
    'vaccine_covid': [
        'vaccine', 'covid', 'pandemic', 'mask', 'booster',
        'pfizer', 'moderna', 'antivax', 'lockdown', 'fauci',
    ],
    'israel_palestine': [
        'israel', 'palestine', 'gaza', 'hamas', 'idf',
        'netanyahu', 'ceasefire', 'zionist', 'hezbollah',
    ],
    'trump': [
        'trump', 'maga', 'indictment', 'mar-a-lago',
        'january 6', 'j6', 'impeach',
    ],
    'immigration': [
        'immigration', 'border', 'migrant', 'asylum',
        'deportation', 'illegal alien', 'refugee', 'caravan',
    ],
    'gun_control': [
        'gun control', 'second amendment', '2nd amendment',
        'shooting', 'nra', 'firearm', 'gun violence',
    ],
    'ALL_POLITICAL': [
        'trump', 'biden', 'democrat', 'republican', 'gop',
        'congress', 'senate', 'election', 'vote', 'ballot',
        'liberal', 'conservative', 'immigration', 'border',
        'abortion', 'gun control', 'vaccine', 'covid',
        'climate change', 'supreme court',
        'israel', 'palestine', 'gaza', 'ukraine', 'russia',
        'woke', 'dei', 'transgender', 'lgbtq', 'censorship',
        'government', 'policy', 'partisan',
    ],
}

# =============================================
# ★ 分析パラメータ
# =============================================
POLARITY_FIRST_N       = 50
MIN_RATING_COUNT       = 20
MIN_RATINGS_TOPIC      = 10
BURST_SPEED_MULTIPLIER = 3.0
BURST_MIN_COUNT        = 5
BURST_THRESHOLD        = None
TREND_MIN_EVALS        = 4
TARGET_TOP_PERCENT     = 25

print(f'▶ チャンク: FILE_CHUNK_INDEX={FILE_CHUNK_INDEX}, ROW_CHUNK_INDEX={ROW_CHUNK_INDEX}')
print(f'  対象ファイル: files[{FILE_OFFSET}:{FILE_OFFSET+FILES_PER_CHUNK}]')
print(f'  行範囲:       [{SKIP_ROWS:,} 〜 {SKIP_ROWS+ROWS_PER_CHUNK:,})')
print(f'  出力サフィックス: {CHUNK_SUFFIX}')
print(f'  例: data/processed/features_v2{CHUNK_SUFFIX}.csv')
print()
print(f'save dir: {SAVE_DIR}')
print(f'topics: {list(TOPICS.keys())}')

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, json

if os.path.exists(DRIVE_DATA):
    print('OK: フォルダが見つかりました')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('共有ドライブ一覧:')
        for d in os.listdir(sd): print(f'  {d}')
    print('マイドライブ:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]: print(f'  {d}')

In [ ]:
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pip install -q statsmodels

In [ ]:
# data/raw/ にデータを準備（あればスキップ、なければ解凍）
raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

# --- ratings: 現在のチャンク分だけ解凍 ---
ratings_folder = os.path.join(DRIVE_DATA, 'RatingData')
if os.path.exists(ratings_folder):
    all_zips = sorted([f for f in os.listdir(ratings_folder) if f.startswith('ratings-') and f.endswith('.zip')])
    chunk_zips = all_zips[FILE_OFFSET:FILE_OFFSET + FILES_PER_CHUNK]
    if not chunk_zips:
        raise RuntimeError(f'FILE_OFFSET={FILE_OFFSET} が ratings zip 数 {len(all_zips)} を超えています')
    needed = set(z.replace('.zip', '.tsv') for z in chunk_zips)
    for f in sorted(os.listdir(raw_dir)):
        if f.startswith('ratings-') and f.endswith('.tsv') and f not in needed:
            print(f'  Removing {f} (not in current chunk)')
            os.remove(os.path.join(raw_dir, f))
    for z in chunk_zips:
        tsv_name = z.replace('.zip', '.tsv')
        if os.path.exists(os.path.join(raw_dir, tsv_name)):
            print(f'  {tsv_name} already exists, skip')
        else:
            print(f'  Unzipping {z} ...')
            subprocess.run(['unzip', '-o', '-q', os.path.join(ratings_folder, z), '-d', raw_dir], check=True)

# --- notes, history: あればスキップ ---
other_targets = [
    ('Notes data', 'notes-'),
    ('Note status history data', 'noteStatusHistory-'),
]
for subfolder, prefix in other_targets:
    folder = os.path.join(DRIVE_DATA, subfolder)
    if not os.path.exists(folder):
        print(f'WARNING: {folder} not found, skip')
        continue
    for f in sorted(os.listdir(folder)):
        if f.startswith(prefix) and f.endswith('.zip'):
            tsv_name = f.replace('.zip', '.tsv')
            if os.path.exists(os.path.join(raw_dir, tsv_name)):
                print(f'  {tsv_name} already exists, skip')
            else:
                print(f'  Unzipping {f} ...')
                subprocess.run(['unzip', '-o', '-q', os.path.join(folder, f), '-d', raw_dir], check=True)

print()
print('=== data/raw/ ===')
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.tsv'):
        size = os.path.getsize(os.path.join(raw_dir, f)) / (1024**3)
        print(f'  {f}  ({size:.2f} GB)')

---
## 1. 全トピックでパイプライン実行 (v2)

In [ ]:
nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_pipeline_v2.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --file-offset {FILE_OFFSET} \
    --skip-rows {SKIP_ROWS} \
    --chunk-suffix {CHUNK_SUFFIX} \
    --polarity-first-n {POLARITY_FIRST_N} \
    --min-rating-count {MIN_RATING_COUNT} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS} \
    --target-top-percent {TARGET_TOP_PERCENT}

---
## 2. トピック別比較 (v2)

In [ ]:
with open('/tmp/topics.json', 'w') as f:
    json.dump(TOPICS, f)

nrows_opt = f'--nrows {NROWS}' if NROWS else ''
burst_thresh_opt = f'--burst-threshold {BURST_THRESHOLD}' if BURST_THRESHOLD is not None else ''

!python scripts/run_by_topic_v2.py \
    {nrows_opt} \
    --max-rating-files {RATINGS_FILES} \
    --file-offset {FILE_OFFSET} \
    --skip-rows {SKIP_ROWS} \
    --chunk-suffix {CHUNK_SUFFIX} \
    --min-ratings {MIN_RATINGS_TOPIC} \
    --topics-json /tmp/topics.json \
    --polarity-first-n {POLARITY_FIRST_N} \
    --burst-speed-multiplier {BURST_SPEED_MULTIPLIER} \
    --burst-min-count {BURST_MIN_COUNT} \
    {burst_thresh_opt} \
    --trend-min-evals {TREND_MIN_EVALS}

---
## 3. 結果の確認 (v2)

In [ ]:
import pandas as pd

df = pd.read_csv(f'data/processed/topic_models_v2{CHUNK_SUFFIX}.csv')
print(f'chunk: {CHUNK_SUFFIX}')
display(df)

In [ ]:
targets = pd.read_csv(f'data/processed/target_notes_v2{CHUNK_SUFFIX}.csv')
print(f'chunk: {CHUNK_SUFFIX}  ターゲットノート数: {len(targets)}')
display(targets.head(20))

In [ ]:
burst_path = f'data/processed/bursts_v2{CHUNK_SUFFIX}.csv'
if os.path.exists(burst_path):
    bursts = pd.read_csv(burst_path)
    print(f'chunk: {CHUNK_SUFFIX}  バースト数: {len(bursts)}')
    print(bursts['burst_type'].value_counts())
else:
    print(f'{burst_path} が存在しません (バースト検出 0 件の可能性)')

---
## 4. 全チャンクの結果をマージ（全チャンクを回し終えたら実行）

v2 用の出力ファイル `*_v2_f*_r*.csv` を統合して `*_v2_all.csv` を作る。

In [ ]:
# v2 用マージ (scripts/merge_chunks.py は v1 ファイル名にハードコードされているのでここで実施)
import glob
import re

PROCESSED = 'data/processed'

def _chunk_tag(name):
    m = re.search(r'(f\d+_r\d+)', name)
    return m.group(1) if m else ''

def merge(pattern, out_name, dedup_on=None):
    paths = sorted(glob.glob(os.path.join(PROCESSED, pattern)))
    if not paths:
        print(f'  [skip] {pattern}: ファイルなし')
        return
    print(f'\n● {pattern} → {out_name}')
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        df['chunk'] = _chunk_tag(os.path.basename(p))
        print(f'    {os.path.basename(p):50s}  {len(df):>8,} rows')
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    if dedup_on:
        keys = [dedup_on] if isinstance(dedup_on, str) else list(dedup_on)
        if all(k in combined.columns for k in keys):
            before = len(combined)
            combined = combined.drop_duplicates(subset=keys, keep='first')
            if before - len(combined):
                print(f'    ⚠ {before-len(combined):,} 件の重複 ({"+".join(keys)}) 除去')
    out = os.path.join(PROCESSED, out_name)
    combined.to_csv(out, index=False)
    print(f'    → {out}  ({len(combined):,} rows)')

merge('bursts_v2_f*_r*.csv',            'bursts_v2_all.csv',            dedup_on='noteId')
merge('target_notes_v2_f*_r*.csv',      'target_notes_v2_all.csv',      dedup_on='noteId')
merge('topic_models_v2_f*_r*.csv',      'topic_models_v2_all.csv',      dedup_on=None)
merge('features_by_topic_v2_f*_r*.csv', 'features_by_topic_v2_all.csv', dedup_on=['topic','noteId'])
merge('features_v2_f*_r*.csv',          'features_v2_all.csv',          dedup_on='noteId')
merge('regression_models_v2_f*_r*.csv', 'regression_models_v2_all.csv', dedup_on=None)

for name in ['topic_models_v2_all.csv', 'features_by_topic_v2_all.csv']:
    path = os.path.join(PROCESSED, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'\n▼ {name}  ({len(df):,} rows)')
        display(df.head(10))

---
## 5. 全データプールでの再回帰 (v2: M0/M1/M2)

`features_by_topic_v2_all.csv` を使ってトピックごとに M0/M1/M2 を再推定する。
M0 → M2 で β_TypeA がどう変わるかを並べる。

In [ ]:
import sys
sys.path.insert(0, '/content/toriumi_x3')
from src.step4_regression_v2.logistic import fit_logistic_regression_v2, fits_to_rows

feat_path = 'data/processed/features_by_topic_v2_all.csv'
if not os.path.exists(feat_path):
    print(f'{feat_path} が存在しません。先にマージセルを実行してください。')
else:
    feats = pd.read_csv(feat_path)
    print(f'読み込み: {len(feats):,} rows ({feats["topic"].nunique()} topics)')
    print(f'bridging_score 利用可: {feats["bridging_score"].notna().sum():,} rows')

    pooled_rows = []
    for topic, sub in feats.groupby('topic', sort=False):
        if len(sub) < 10 or sub['deleted'].nunique() < 2 or sub['type_a'].nunique() < 2:
            pooled_rows.append({'topic': topic, 'model': 'all', 'n': len(sub),
                                'note': 'skip: insufficient variance'})
            continue
        fits = fit_logistic_regression_v2(sub, print_results=False)
        pooled_rows.extend(fits_to_rows(fits, topic=topic))

    pooled_df = pd.DataFrame(pooled_rows)
    pooled_df.to_csv('data/processed/topic_models_v2_pooled.csv', index=False)
    print()
    print('=== プール再回帰 (v2) ===')
    display(pooled_df)

    # M0 vs M2 の β_typeA 並列表示
    if 'beta_type_a' in pooled_df.columns:
        pivot = pooled_df.pivot_table(index='topic', columns='model', values='beta_type_a')
        print('\n=== β_TypeA across models ===')
        display(pivot)

---
## 6. 結果を Drive に保存

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
for f in os.listdir('data/processed'):
    src_path = os.path.join('data/processed', f)
    if (f.endswith('.csv') or f.endswith('.png')) and ('_v2' in f):
        subprocess.run(['cp', src_path, SAVE_DIR])
print(f'Done! Results saved to: {SAVE_DIR}')